In [29]:
import sys
sys.path.append('../preprocess')
from qbiolip import load_db
import glob
import torch
import re
import pandas
from plotly import express as xp
from plotly import graph_objects as go

qbiolip_path = "../../databases/QBioLiP/qbiolip.csv"
ec_uniprot_ids_path = "../../databases/ec_uniprot_ids.txt"

df = load_db(qbiolip_path, ec_uniprot_ids_path)
df = df.loc[df.target]
output_paths = glob.glob("../../databases/NeuralPLexer/inferences/*")

targets = []
preds = []
names = []
ranks = []

# fetch outputs:
for p in output_paths:
    outputs = glob.glob(f"{p}/*.sdf")
    if len(outputs) == 0:
        continue
    names.append(p.split("/")[-1])
    output_ranks = []
    output_preds = []
    for o in outputs:
        filename = o.split("/")[-1]
        if filename == "lig_all.sdf":
            continue
        with open(o) as sdf_f:
            lines = [l.split() for l in sdf_f]
        lines = [l for l in lines if len(l) == 16] #only xyz coordinates lines
        xyz = torch.tensor([
            [
                float(l[0]), #x
                float(l[1]), #y
                float(l[2]), #z
            ]
        for l in lines])
        if filename == "lig_ref.sdf":
            targets.append(xyz)
        else:
            output_preds.append(xyz)
            output_ranks.append(int(re.search(r"(?<=lig_rank)[0-9]{1,2}(?=_)", filename).group()))
    output_ranks = torch.tensor(output_ranks)
    output_preds = torch.stack(output_preds)
    ranks_sorted = output_ranks.argsort(descending=False) # the lower the better
    # sort by rank:
    output_preds = output_preds[ranks_sorted]
    # save in tensors:
    preds.append(output_preds) 
    ranks.append(ranks_sorted)

# calculate metrics:
rank = 0 #look at the highest confidence score
print(len(names))
for i in range(len(names)):
    if preds[i].shape[0] == 0:
        continue
    pred = preds[i][rank]
    target = targets[i]
    rmsd = (pred-target).pow(2).sum(dim=1).mean().sqrt()
    name = names[i]
    col = ("neuralplexer_rmsd", "neuralplexer_trimed_rmsd")["_trimed" in name]
    df.loc[df.ligand_detail == name.replace("_trimed", ""), [col]] = rmsd
print("Number of available rmsd for full receptor:", df.loc[~df.neuralplexer_rmsd.isna()].shape[0])
print("Number of available trimed rmsd:",df.loc[~df.neuralplexer_trimed_rmsd.isna()].shape[0])

# plot metrics:
neuralplexer_df = df.loc[(~df.neuralplexer_rmsd.isna()) & (~df.neuralplexer_trimed_rmsd.isna())]

# plot untrimed:
neuralplexer_df.sort_values(by="neuralplexer_rmsd", inplace=True)
neuralplexer_df.reset_index(drop=True,inplace=True)
colors = pandas.Series(["red"]*neuralplexer_df.shape[0])
colors[neuralplexer_df.loc[neuralplexer_df.group=="B"].index] = "blue"
colors[neuralplexer_df.loc[neuralplexer_df.group=="C"].index] = "green"

fig = xp.bar(
    neuralplexer_df, x="ligand_detail", y="neuralplexer_rmsd",
    title="neuralplexer RMSD on 90 enzyme-ligand complexes with implicit hydrogenation, true blind docking, top 1",
    color=colors,
    color_discrete_map="identity",
    category_orders={"ligand_detail": neuralplexer_df.ligand_detail.tolist()},
    log_y=True,
)
fig.update_layout(
    xaxis_title="Name of the complex",
    yaxis_title="RMSD, Logarithmic scale",
)
fig.show()

# plot trimed
neuralplexer_df.sort_values(by="neuralplexer_trimed_rmsd", inplace=True)
neuralplexer_df.reset_index(drop=True,inplace=True)
colors = pandas.Series(["red"]*neuralplexer_df.shape[0])
colors[neuralplexer_df.loc[neuralplexer_df.group=="B"].index] = "blue"
colors[neuralplexer_df.loc[neuralplexer_df.group=="C"].index] = "green"

fig = xp.bar(
    neuralplexer_df, x="ligand_detail", y="neuralplexer_trimed_rmsd",
    title="neuralplexer RMSD on 90 enzyme-ligand complexes with implicit hydrogenation, pseudo blind docking, top 1",
    color=colors,
    color_discrete_map="identity",
    category_orders={"ligand_detail": neuralplexer_df.ligand_detail.tolist()},
    log_y=True,
)
fig.update_layout(
    xaxis_title="Name of the complex",
    yaxis_title="RMSD, Logarithmic scale",
)
fig.show()
            
# plot rmsd per group # trimed
x = neuralplexer_df.group.unique()
y_trimed = [neuralplexer_df.loc[neuralplexer_df.group==group].neuralplexer_trimed_rmsd.mean() for group in x]
# plot rmsd per group # not trimed
y_untrimed = [neuralplexer_df.loc[neuralplexer_df.group==group].neuralplexer_rmsd.mean() for group in x]
fig = go.Figure([
    go.Bar(x=x, y=y_trimed, name="trimed"),
    go.Bar(x=x, y=y_untrimed, name="untrimed"),
])
fig.update_layout(
    title="Mean RMSD for each categories"
)
fig.show()

Len of the database: 25415
Columns: id, pdb_id, assembly_id, stoichiometry, resolution, uniprot_id, assembly_detail, ligand_id, site, ligand_detail, pubmed_id, interaction, relevant, binding_site, binding_site_pdb, bindingMOAD, pdbbind_cn, binding_db, group, lig_canon_smiles, lig_atoms, rec_len_resi, target, trimed_rec_path
134
Number of available rmsd for full receptor: 48
Number of available trimed rmsd: 86


/tmp/ipykernel_70650/3534006667.py:78: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



/tmp/ipykernel_70650/3534006667.py:99: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

